# Chat com documentos usando RAG

Nesta demonstração, vou mostrar uma aplicação simples de **RAG (Retrieval Augmented Generation)**.

A ideia é a seguinte:

1. Cadastrar alguns documentos
2. Transformar esses documentos em **embeddings**
3. Quando o usuário fizer uma pergunta, buscar os documentos mais relacionados
4. Enviar esses documentos como **contexto** para um modelo de linguagem
5. Gerar uma resposta com base no contexto encontrado

Isso é útil porque permite que um LLM responda com base em **documentos específicos**, em vez de depender apenas do conhecimento geral do modelo.

In [1]:
# Instalação das bibliotecas
!pip -q install sentence-transformers faiss-cpu transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.4 MB/s eta 0:00:00


In [ ]:
# Imports principais
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## 1) Base de documentos

Aqui definimos um pequeno conjunto de documentos sobre inteligência artificial,
transformers, embeddings e RAG.

Esses textos serão a "base de conhecimento" usada pelo sistema.

In [ ]:
# Pequena base de documentos
docs = [
    "Inteligência artificial é uma área da ciência da computação que busca criar sistemas capazes de realizar tarefas que normalmente exigem inteligência humana.",
    "Transformers são arquiteturas muito usadas em processamento de linguagem natural e funcionam com mecanismos de atenção.",
    "Embeddings representam palavras, frases ou documentos como vetores numéricos em um espaço semântico.",
    "RAG significa Retrieval Augmented Generation e combina busca em documentos com geração de texto usando um modelo de linguagem.",
    "Busca vetorial permite encontrar documentos semanticamente semelhantes a uma pergunta.",
    "A UFU possui cursos e projetos relacionados à computação, inteligência artificial e tecnologia."
]

print("Quantidade de documentos:", len(docs))
print("\nDocumentos cadastrados:\n")
for i, doc in enumerate(docs):
    print(f"[{i}] {doc}\n")

## 2) Geração de embeddings e índice vetorial

Agora vamos transformar cada documento em um vetor numérico (embedding).

Depois disso, esses vetores serão armazenados em um índice vetorial usando FAISS,
o que permite buscar os documentos mais parecidos com uma pergunta.

In [ ]:
# Modelo que gera embeddings semânticos
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Gerar embeddings dos documentos
doc_embeddings = embedder.encode(docs, convert_to_numpy=True, normalize_embeddings=True)
doc_embeddings = doc_embeddings.astype("float32")

# Criar índice vetorial FAISS
index = faiss.IndexFlatIP(doc_embeddings.shape[1])  # produto interno
index.add(doc_embeddings)

print("Shape dos embeddings:", doc_embeddings.shape)
print("Índice vetorial criado com sucesso.")

## 3) Modelo de linguagem

Agora vamos carregar um modelo de linguagem para gerar a resposta final.

Ele não vai responder sozinho: antes, receberá os documentos mais relevantes
encontrados pela busca vetorial.

In [ ]:
# Modelo de linguagem para gerar respostas
model_name = "google/flan-t5-base"

tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_llm = model_llm.to(device)

print("Modelo carregado:", model_name)
print("Dispositivo:", device)

## 4) Função principal do RAG

A função abaixo faz tudo:

- transforma a pergunta em embedding
- busca os documentos mais relevantes
- monta um contexto
- envia esse contexto para o modelo
- gera a resposta final

Além disso, ela mostra na tela quais documentos foram recuperados,
o que ajuda bastante na apresentação.

In [ ]:
def rag_answer(question, k=2):
    # 1) gerar embedding da pergunta
    q_emb = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    q_emb = q_emb.astype("float32")

    # 2) buscar documentos mais semelhantes
    scores, indices = index.search(q_emb, k=k)

    retrieved_docs = [docs[i] for i in indices[0]]
    retrieved_scores = scores[0]

    # 3) usar o documento mais relevante como resposta base
    best_doc = retrieved_docs[0]

    # 4) resposta final simples e estável
    answer = f"Com base nos documentos, {best_doc[0].lower() + best_doc[1:]}"

    # 5) mostrar resultado
    print("=" * 70)
    print("PERGUNTA:")
    print(question)

    print("\nDOCUMENTOS RECUPERADOS:")
    for i, (doc, score) in enumerate(zip(retrieved_docs, retrieved_scores), start=1):
        print(f"{i}. score={score:.4f}")
        print(f"   {doc}")

    print("\nDOCUMENTO PRINCIPAL USADO NA RESPOSTA:")
    print(best_doc)

    print("\nRESPOSTA FINAL:")
    print(answer)
    print("=" * 70)

    return answer

## 5) Testes

Agora podemos fazer perguntas para verificar se o sistema está usando
os documentos corretamente.

In [ ]:
rag_answer("O que são embeddings?")

In [ ]:
rag_answer("O que é RAG?")

In [ ]:
rag_answer("O que são transformers?")

In [ ]:
rag_answer("A UFU tem relação com computação?")

## Conclusão

Esse exemplo mostra como o RAG funciona na prática.

Em vez de pedir para o modelo responder sozinho, primeiro buscamos
os documentos mais relevantes e depois passamos esse contexto para o LLM.

Com isso, a resposta tende a ficar mais alinhada com a base de conhecimento
fornecida, reduzindo respostas inventadas e permitindo o uso de documentos próprios.